# Pretrained Models & Fine-tuning

Reach for this when you need: 
- Reference for loading SOTA models from `torchvision` or `timm`.
- To implement Transfer Learning (freezing/unfreezing layers).
- To replace classification heads for custom datasets.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Loading Pretrained Weights

| Library | Description | Usage |
| :--- | :--- | :--- |
| `torchvision.models` | Mainstream CV models (ResNet, VGG) | Standard baselines |
| `timm` | SOTA Computer Vision models (ViT, ConvNeXt) | Production-scale SOTA models |

In [ ]:
# Torchvision way (Modern Weights Enum)
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# TIMM way (String Identifier)
model_timm = timm.create_model('resnet50', pretrained=True, num_classes=10)

## 2. Freezing & Replacing Heads

Freezing prevents the backbone weights from being updated by the optimizer.

✅ **Use when**: Fine-tuning on a very small dataset (e.g. 100 images).
❌ **Don't use when**: High-quality data is abundant (better to fine-tune the whole model).

In [ ]:
model = models.resnet18(weights='DEFAULT')

# 1. Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# 2. Replace head (automatically has requires_grad=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5) # 5 custom classes

## 3. Discriminative Learning Rates

Different learning rates for different parts of the model (backbone vs. head).

✅ **Use when**: Unfreezing the model after initial training of the head.
❌ **Don't use when**: Training from scratch.

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5}, # Fine-tune top layers slowly
    {'params': model.fc.parameters(), 'lr': 1e-3}     # Train head faster
], weight_decay=1e-2)

### Common Pitfalls
- **Statefulness**: If you replace the head `model.fc`, ensure you MOVE the model to the device AFTER replacement, or move the new layer specifically.
- **Requires Grad**: Double-check `model.parameters()` if the loss isn't decreasing; frequently, a frozen backbone is the culprit.
- **Input Size**: Some architectures like ResNet are input-size agnostic to a degree, but others (e.g. ViT) often require specific image dimensions (224x224).

### Key Takeaways
- Transfer Learning is the default approach for >95% of industry CV tasks.
- `timm` is usually the preferred library for the latest models (Vision Transformers, etc).
- Always start with a frozen backbone and only unfreeze top layers as needed.